In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, FilterCatalog

rosemary = pd.read_csv("../data/cultivar_compound_catalogue.csv").drop_duplicates("compound_id")

params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)

dev_results = []
for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    rotb = Lipinski.NumRotatableBonds(mol)
    psa = rdMolDescriptors.CalcTPSA(mol)
    
    lipinski = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    veber = (rotb <= 10) and (psa <= 140)
    pains = not pains_catalog.HasMatch(mol)
    agg = True # Placeholder locked rule
    
    dev_score = sum([lipinski, veber, pains, agg]) / 4.0
    
    dev_results.append({
        "compound_id": row["compound_id"],
        "developability_score": dev_score,
        "lipinski": lipinski,
        "veber": veber,
        "pains": pains,
        "aggregator": agg
    })

pd.DataFrame(dev_results).to_csv("../results/developability_scores.csv", index=False)
print("Developability scores saved.")

Developability scores saved.
